# Tool Test Demo

验证 `agent_tools.py` 全部 13 个工具可正常调用。

**测试前提**：在 BigQuant 环境运行，`run_data_probe()` 和 `load_etf_data_bigquant()` 依赖平台数据表。

In [ ]:
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "src").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"project_root: {p}")
        break

from src.agent_tools import list_agent_tools, call_agent_tool
from src.config import DEFAULT_CONFIG

config = DEFAULT_CONFIG

## 1. 工具自检

In [ ]:
tools = list_agent_tools()
print(f"registered tools ({len(tools)}):")
for t in tools:
    print(f"  - {t}")

## 2. 解析型工具（不需数据，直接可测）

In [ ]:
intent = call_agent_tool('resolve_factor_intent',
    query='成交额放大、趋势走强、相对沪深300更强的行业ETF')
print('research_type:', intent['research_type'])
print('matched_factors:', [f['name'] for f in intent['matched_factors']])
print('factor_plan:', [f['name'] for f in intent['factor_plan']])
print('conditions:', intent['conditions'])

plan = call_agent_tool('build_factor_plan',
    user_idea='OBV上涨，缩量下跌，趋势走强')
print('\nplan_payload:', plan)

## 3. 字段探测（需 BigQuant 环境）

In [ ]:
probe = call_agent_tool('run_data_probe')
print(f"probed tables: {list(probe.keys())}")

## 4. 数据加载 + 因子分析

用 `load_etf_data` 加载数据，再用 `run_factor_analysis` 做 IC 检验。

In [ ]:
df = call_agent_tool('load_etf_data',
    start_date=config.start_date,
    end_date=config.end_date,
    table_name=config.fund_table,
)
print(f"data shape: {df.shape}")
print(f"columns: {list(df.columns)}")

In [ ]:
# 检查可用因子列
available = call_agent_tool('get_available_factors', df=df)
print(f"available factors: {available}")

factor_cols = [c for c in config.factor_cols if c in df.columns]
result = call_agent_tool('run_factor_analysis',
    df=df, factor_cols=factor_cols, target_cols=config.target_cols)
print('factor analysis keys:', list(result.keys()))
if 'ic_summary' in result:
    print(result['ic_summary'])

## 5. 打分 + 回测 + 诊断

In [ ]:
weight_schemes = call_agent_tool('build_weight_schemes',
    factor_cols=factor_cols, factor_analysis_result=result)
print('weight_schemes keys:', list(weight_schemes.keys()))
print('hypothesis_weight:', weight_schemes['hypothesis_weight'])

scored = call_agent_tool('compute_composite_score',
    df=df, factor_metadata=None, weights=weight_schemes['hypothesis_weight'])
print('composite_score' in scored.columns)

In [ ]:
backtest = call_agent_tool('diagnose_strategy',
    factor_analysis_result=result, backtest_result=None, weights=weight_schemes['hypothesis_weight'])
print('diagnosis:', backtest)

## 6. 条件概率研究

In [ ]:
cond_result = call_agent_tool('run_conditional_probability_test',
    df=df,
    conditions=[{'col': 'amount_ratio_20d', 'op': '>', 'value': 1.2},
                {'col': 'trend_strength', 'op': '>', 'value': 0}],
    target='future_return_5d',
    min_event_count=200,
)
print('conditional_probability:', cond_result.get('conditional_probability'))
print('baseline_probability:', cond_result.get('baseline_probability'))

## 7. 全流程 Pipeline

一键跑完整因子研究和条件概率研究。

In [ ]:
pipe = call_agent_tool('run_factor_research_pipeline',
    user_idea='成交额放大、趋势走强、相对沪深300更强的行业ETF',
    start_date=config.start_date,
    end_date=config.end_date,
    table_name=config.fund_table,
)
print('pipeline keys:', list(pipe.keys()))
print('data_shape:', pipe.get('data_shape'))
print('backtest_performance:', pipe.get('backtest_result', {}).get('performance'))
print('factor_result keys:', list(pipe.get('factor_result', {}).keys()))

In [ ]:
cond_pipe = call_agent_tool('run_condition_research_pipeline',
    user_idea='当单日量比 > 1、换手率 < 5、今日上涨时，明天上涨几率大吗？',
    start_date=config.start_date,
    end_date=config.end_date,
    table_name=config.fund_table,
)
print('condition_pipeline keys:', list(cond_pipe.keys()))
print('conditional_probability:', cond_pipe.get('result', {}).get('conditional_probability'))
print('baseline_probability:', cond_pipe.get('result', {}).get('baseline_probability'))